# end-grad-default-ones-like — ex3: resolve end_grad on a 0-D scalar loss-tip

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `end-grad-default-ones-like`. Running the final beacon cell reports progress against the `Backprop: end-grad ones_like default` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: end-grad ones_like default` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`end-grad-default-ones-like`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "end-grad-default-ones-like"
DD_SUBTOPIC = "Backprop: end-grad ones_like default"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## end_grad on a 0-D scalar end-node — the loss-tip case

Ex1 covered the default-vs-explicit branch on a multi-element tensor; ex2 used a `(B,)` per-sample-weighted loss. The deepening move handles the MOST COMMON real case: the end-node is the loss itself — a 0-D scalar.

```python
loss = (x.array ** 2).sum()         # 0-D scalar — shape ()
end_node = MiniTensor(loss, ...)

# Default path: ones_like(scalar) -> tensor(1.0), shape ()
end_grad = t.ones_like(end_node.array)
assert end_grad.shape == ()
assert end_grad.item() == 1.0
```

**Why scalar end-grad always seeds to 1.0.** Calling `loss.backward()` in real torch corresponds to `dL/dL = 1` — the identity. Our manual seed is the same: `ones_like` on a scalar gives `tensor(1.0)`, exactly the chain-rule identity.

**The shape-mismatch failure mode.** If a caller PASSES an explicit `end_grad` of the wrong rank — e.g. `(B,)` against a 0-D end-node — the resolver must raise an `AssertionError`. Silently broadcasting is wrong: it would scale every leaf by `B` copies of the seed and break gradient accounting.

### Exercise 3 — resolve end_grad on a 0-D scalar loss-tip

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the .backward() entry-point convention to a 0-D scalar end-node: ones_like gives tensor(1.0), an explicit 0-D end_grad is unboxed as-is, and a non-scalar explicit end_grad raises AssertionError.
> Keywords: end-grad, scalar, 0-d, loss-tip, ones_like
> ```

**KCs targeted:** `scalar-end-grad-is-ones`, `shape-mismatch-raises`

Implement `ex3_resolve_scalar_end_grad(end_node, end_grad)`.

Inputs:
- `end_node`: a `MiniTensor` wrapping a 0-D (scalar) `torch.Tensor` — this is the typical loss tensor.
- `end_grad`: either `None` OR a `MiniTensor`.

Behaviour:

1. If `end_grad is None` — default-path. Return `t.ones_like(end_node.array)`. For a scalar end-node this is `tensor(1.0)` with shape `()`.
2. If `end_grad` is a `MiniTensor` — explicit path. First assert `end_grad.array.shape == end_node.array.shape` with a helpful message that names both shapes. If they match, return `end_grad.array`.
3. Output type: raw `torch.Tensor` (NOT a MiniTensor).

Constraints:
- Use `t.ones_like` for the default, NOT `t.tensor(1.0)`. `ones_like` preserves `device` and `dtype`.
- The assertion failure must be an `AssertionError` (not a plain `ValueError`). The tests catch that specifically.
- Do NOT call `torch.autograd`.

In [ ]:
def ex3_resolve_scalar_end_grad(end_node, end_grad):
    """Resolve end_grad for a 0-D scalar end-node. Returns raw torch.Tensor."""
    raise NotImplementedError()


def _test_ex3():
    # --- default path: 0-D end-node, end_grad=None -> ones_like (scalar 1.0) ---
    x = t.tensor(2.5)            # 0-D float scalar
    end_node = MiniTensor(x, requires_grad=True)
    seed = ex3_resolve_scalar_end_grad(end_node, None)
    assert isinstance(seed, t.Tensor), f'default seed must be a torch.Tensor; got {type(seed).__name__}'
    assert seed.shape == (), f'scalar end-node -> scalar seed; got shape {seed.shape}'
    assert seed.item() == 1.0, f'ones_like(scalar) == 1.0; got {seed.item()}'
    assert seed.dtype == x.dtype, f'dtype should be preserved; got {seed.dtype} vs {x.dtype}'

    # --- explicit path: 0-D end_grad MiniTensor of matching shape -> unboxed ---
    explicit = MiniTensor(t.tensor(7.5))
    seed = ex3_resolve_scalar_end_grad(end_node, explicit)
    assert isinstance(seed, t.Tensor)
    assert seed.shape == ()
    assert seed.item() == 7.5, f'explicit value passes through; got {seed.item()}'

    # --- shape mismatch: end_grad is (B,) against a 0-D end-node -> AssertionError ---
    mismatched = MiniTensor(t.ones(3))
    raised = False
    try:
        ex3_resolve_scalar_end_grad(end_node, mismatched)
    except AssertionError as e:
        raised = True
        msg = str(e)
        # The message must name BOTH shapes for diagnosis.
        assert 'torch.Size([])' in msg or '()' in msg or 'shape' in msg.lower(), (
            f'assertion message should mention shapes; got {msg!r}'
        )
    assert raised, 'shape mismatch must raise AssertionError'

    # --- another shape mismatch: 2-D end_grad against 0-D end-node ---
    raised = False
    try:
        ex3_resolve_scalar_end_grad(end_node, MiniTensor(t.ones(2, 3)))
    except AssertionError:
        raised = True
    assert raised, '(2,3) vs () must also raise'

    # --- preserves dtype on non-default scalar dtypes (float64) ---
    x64 = t.tensor(3.14, dtype=t.float64)
    end_node64 = MiniTensor(x64, requires_grad=True)
    seed64 = ex3_resolve_scalar_end_grad(end_node64, None)
    assert seed64.dtype == t.float64, f'dtype preserved through ones_like; got {seed64.dtype}'
    assert seed64.item() == 1.0

    # --- semantic check: composed with log_back at the loss tip ---
    # end_node = log(x_leaf).sum() — 0-D scalar. seed=ones_like -> 1.0. 
    # Then dL/dx_leaf = seed * (1/x_leaf) — should match torch.autograd.
    x_leaf_raw = t.tensor([1.0, 2.0, 4.0], requires_grad=True)
    loss = t.log(x_leaf_raw).sum()
    loss_mt = MiniTensor(loss.detach(), requires_grad=True)
    seed = ex3_resolve_scalar_end_grad(loss_mt, None)
    assert seed.item() == 1.0
    # Chain one step (this is the integration with log_back):
    # dL/dx_leaf = seed * (1/x_leaf) — but seed is 0-D, must broadcast over (3,).
    hand_grad = seed * (1.0 / x_leaf_raw.detach())
    loss.backward()
    assert t.allclose(hand_grad, x_leaf_raw.grad, atol=1e-6), (
        f'hand-rolled scalar seed must match torch; got hand={hand_grad} torch={x_leaf_raw.grad}'
    )
    _dd_passed.add('ex3')
    print("ex3 ✓")

_test_ex3()

<details><summary>Solution</summary>

```python
def ex3_resolve_scalar_end_grad(end_node, end_grad):
    if end_grad is None:
        return t.ones_like(end_node.array)
    assert end_grad.array.shape == end_node.array.shape, (
        f'end_grad shape {tuple(end_grad.array.shape)} != '
        f'end_node shape {tuple(end_node.array.shape)}'
    )
    return end_grad.array
```

**`ones_like` not `torch.ones(end_node.array.shape)`.** `ones_like` preserves `device` AND `dtype`. Hardcoding shape+dtype defeats the purpose — if `end_node` is on CUDA float64, the seed must be CUDA float64 too, otherwise the first multiplication in the reverse pass triggers a device or dtype mismatch.

**Why `AssertionError` not `ValueError`.** The autograd entry point treats shape mismatch as a CALLER BUG — they passed an `end_grad` that doesn't match. `assert` is the right form: it's a contract check, not a domain error. Real PyTorch raises `RuntimeError` instead, but for the manual-autograd drill, `AssertionError` is the convention used across the rest of the chain.

**Scalar broadcast in the chain.** A 0-D seed multiplied by a non-scalar local grad (e.g. `1/x` where x is (3,)) broadcasts via PyTorch's normal rules — the seed scales every element uniformly. That's exactly what `loss.backward()` does in real torch.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()